# 04 — Reproducible Temporal Knowledge Graph (v2)

**Inputs:** frozen Entity Resolution exports; original Jira ticket JSON; optional original PRD and *actual* meeting transcript files.  
**Outputs:** reproducible graph JSONL, Neo4j CSV and Cypher, temporal/provenance diagnostics, checkpoint interface, handoff manifest.  
**No Neo4j server, APOC, LLM calls or re-extraction required in Colab.**

The old graph is **not** used as input. All graph nodes and edges are reconstructed from the frozen ER records; Jira and original document metadata enrich the graph. The baseline graph remains a comparison only.

### Temporal contract
- `event_time`: when the event happened (may be earlier than disclosure).
- `observed_at`: earliest **defensible availability time** of the assertion in the selected source. Not the time of notebook execution.
- `valid_from` / `valid_to`: source-supported fact validity, not guessed.
- `temporal_status`: `EXACT`, `DERIVED`, `UNKNOWN`.
- `temporal_basis`: explains the source and derivation.

**Conservative rule:** Unknown-availability facts are excluded from strict point-in-time snapshots. Jira issue creation does **not** imply that an undated dependency was already known.


## 1. Colab setup — export only
No Neo4j credentials or running database required. Change `OUTPUT_ROOT` to the folder containing the three frozen ER exports.

In [1]:
from pathlib import Path
import os, json, csv, re, hashlib, collections
from datetime import datetime, timezone
import pandas as pd
from tqdm.auto import tqdm

try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
except ImportError:
    userdata = None

OUTPUT_ROOT = Path('/content/drive/MyDrive/GC1/entity_resolution_output')
KG_ROOT = OUTPUT_ROOT / 'temporal_kg_output'
KG_ROOT.mkdir(parents=True, exist_ok=True)
CANONICAL_FILE = OUTPUT_ROOT / 'canonical_entities.jsonl'
RESOLVED_FILE = OUTPUT_ROOT / 'resolved_extraction_records.jsonl'
MAP_FILE = OUTPUT_ROOT / 'entity_resolution_map.jsonl'
for path in (CANONICAL_FILE, RESOLVED_FILE, MAP_FILE):
    if not path.is_file(): raise FileNotFoundError(f'Missing ER handoff: {path}')
print('Input directory:', OUTPUT_ROOT)


Mounted at /content/drive
Input directory: /content/drive/MyDrive/GC1/entity_resolution_output


## 2. Read and validate the authoritative ER handoff

In [2]:
def read_jsonl(path):
    with open(path, encoding='utf-8') as f:
        for line_number, line in enumerate(f, 1):
            if line.strip():
                try: yield json.loads(line)
                except Exception as exc: raise ValueError(f'{path}:{line_number}: {exc}') from exc

canonical = list(read_jsonl(CANONICAL_FILE))
resolved = list(read_jsonl(RESOLVED_FILE))
resolution_map = list(read_jsonl(MAP_FILE))
ids = [x['canonical_id'] for x in canonical]
assert len(ids) == len(set(ids)), 'Canonical IDs are not unique'
entity_by_id = {x['canonical_id']:x for x in canonical}
assert all(x.get('instance_id') for x in canonical)
assert len({(x['instance_id'], x['source_type'], x['source_id'], x['local_id']) for x in resolution_map}) == len(resolution_map)
assert all(x['canonical_id'] in entity_by_id for x in resolution_map)
print({'canonical_entities':len(canonical),'resolved_records':len(resolved),'resolution_map':len(resolution_map)})


{'canonical_entities': 40325, 'resolved_records': 5827, 'resolution_map': 61392}


## 3. Time model and graph contract
- `event_time`: when the reported event occurred.
- `valid_from` / `valid_to`: stated effective interval (end exclusive).
- `observed_at`: earliest **defensible** time the information was available; a source publication timestamp is used when present.
- `source_timestamp`: original source timestamp.
- Missing or unparseable times remain `null`, **not** replaced with the current date.
- `:Fact` nodes reify relationships, allowing individual evidence and time intervals even when multiple sources assert the same edge. `:RELATED_TO` is intentionally **not** materialized as an unconditional historical edge.
- Temporal statements are *not* inferred from prose: fields absent from extraction remain unknown.
- `:Evidence` nodes are scoped to a source record and linked to all supported facts in that record; this is record-level provenance, not a claim that a specific quote supports every fact.

In [3]:
from dateutil import parser as dateparser

def iso(value):
    if value is None or value == '': return None
    if isinstance(value, (dict,list)): return None
    try:
        dt = dateparser.isoparse(str(value))
        if dt.tzinfo is None: return None  # never invent a timezone
        return dt.astimezone(timezone.utc).isoformat(timespec='seconds')
    except (ValueError,TypeError,OverflowError): return None

def first_time(obj, keys):
    if not isinstance(obj,dict): return None
    for k in keys:
        t = iso(obj.get(k))
        if t: return t
    return None

def digest(*parts):
    payload=json.dumps(parts, ensure_ascii=False, sort_keys=True, default=str)
    return hashlib.sha256(payload.encode()).hexdigest()[:24]

def compact(obj): return json.dumps(obj, ensure_ascii=False, sort_keys=True, default=str)

def safe_type(s):
    s=re.sub('[^A-Z0-9_]', '_', str(s or 'RELATED_TO').upper())
    return s if re.fullmatch('[A-Z][A-Z0-9_]*',s) else 'RELATED_TO'

SECTIONS={'events':'Event','decisions':'Decision','action_items':'ActionItem','blockers':'Blocker','dependencies':'Dependency'}
REFERENCE_KEYS={'source_entity','target_entity','entity_id','canonical_id','issue_id','requirement_id','decision_id','blocker_id','dependency_id','action_item_id','related_entities','affected_entities','entities','owner','assignee','raised_by','made_by','blocked_by','depends_on','source','target'}


## 4 — Discover original source documents and Jira tickets

Set the two roots below. Files are discovered recursively. Meeting `*.prompt.txt` files are excluded. `all_300_tickets_full.json` is preferred over other Jira variants to avoid loading the same tickets twice. Project instance mapping uses exact `NEXUS-I01` style identifiers, never folder order alone. For Jira-only records, a unique project-key match is required; ambiguous instances are flagged.

**If multiple instances share a Jira project key:** set `JIRA_FOLDER_INSTANCE_MAP` explicitly. Unmapped tickets are reported, not silently assigned.


In [4]:
from pathlib import Path
import re, json, collections
JIRA_ROOT = Path('/content/drive/MyDrive/GC1/GC1_Jira_Dataset')
DOCUMENT_ROOTS = [JIRA_ROOT, Path('/content/drive/MyDrive/GC1')]
JIRA_FOLDER_INSTANCE_MAP = {}  # Example: {'01_JBEAP': 'JBEAP-I01'}; fill when project key has multiple instances
REQUIRE_JIRA_MATCH = False   # Set True after confirming source folder mapping

instance_ids = sorted({x['instance_id'] for x in canonical})
instances_by_key = collections.defaultdict(list)
for inst in instance_ids:
    instances_by_key[inst.split('-I')[0].upper()].append(inst)

jira_paths = sorted(JIRA_ROOT.rglob('all_300_tickets_full.json')) if JIRA_ROOT.exists() else []
print('Jira project files discovered:', len(jira_paths))
if not jira_paths: print('WARNING: no original Jira files discovered; verify JIRA_ROOT')

def read_tickets(path):
    data=json.loads(path.read_text(encoding='utf-8'))
    if isinstance(data,list): return data
    if isinstance(data,dict):
        for key in ('tickets','issues','records','data'):
            if isinstance(data.get(key),list): return data[key]
    raise ValueError(f'Unexpected ticket structure: {path}')

def path_instance(path):
    matches=set(re.findall(r'([A-Za-z][A-Za-z0-9_]*-I\d{2,3})',str(path),re.I))
    matches={m.upper() for m in matches} & set(instance_ids)
    return next(iter(matches)) if len(matches)==1 else None

jira_issues={}; jira_diagnostics=[]
for path in jira_paths:
    tickets=read_tickets(path)
    hinted=path_instance(path) or JIRA_FOLDER_INSTANCE_MAP.get(path.parent.name)
    for ticket in tickets:
        if not isinstance(ticket,dict): continue
        key=str(ticket.get('project_key') or (str(ticket.get('issue_key','')).split('-')[0])).upper()
        candidates=instances_by_key.get(key,[])
        inst=hinted if hinted in candidates else (candidates[0] if len(candidates)==1 else None)
        if not inst:
            jira_diagnostics.append({'file':str(path),'issue_key':ticket.get('issue_key'),'problem':'ambiguous_or_unmapped_instance','candidates':candidates})
            continue
        issue_key=str(ticket.get('issue_key') or '').upper()
        if not issue_key:continue
        pair=(inst,issue_key)
        if pair in jira_issues and jira_issues[pair]!=ticket:
            jira_diagnostics.append({'file':str(path),'issue_key':issue_key,'problem':'conflicting_duplicate_issue'})
            continue
        jira_issues[pair]=ticket
print('Mapped Jira tickets:',len(jira_issues),'diagnostics:',len(jira_diagnostics))
if REQUIRE_JIRA_MATCH: assert jira_issues, 'No Jira tickets mapped; correct folder names or add explicit mapping'

# Document metadata: do not parse prompt context as actual meeting evidence.
documents={}; doc_diagnostics=[]
for root in DOCUMENT_ROOTS:
    if not root.exists():continue
    for path in root.rglob('*.txt'):
        if '.prompt.' in path.name.lower():continue
        m=re.match(r'^([A-Za-z][A-Za-z0-9_]*-I\d{2,3})-(M\d+|PRD-V\d+)\.txt$',path.name,re.I)
        if not m:continue
        inst=m.group(1).upper()
        if inst not in instance_ids:continue
        text=path.read_text(encoding='utf-8',errors='replace')
        kind='MEETING' if m.group(2).upper().startswith('M') else 'PRD'
        if kind=='MEETING' and not re.search(r'^Transcript\s*:',text,re.M|re.I):
            doc_diagnostics.append({'file':str(path),'problem':'not_a_transcript'});continue
        source_id=f'{inst}-{m.group(2).upper()}'
        if kind=='MEETING':
            mt=re.search(r'^Date/time:\s*(.+)$',text,re.M|re.I)
        else:
            mt=re.search(r'^\*\*Document date:\*\*\s*(.+)$',text,re.M|re.I)
        raw_date=mt.group(1).strip() if mt else None
        # A date-only PRD becomes available at end of that UTC day (conservative).
        if raw_date and re.fullmatch(r'\d{4}-\d{2}-\d{2}',raw_date): raw_date+='T23:59:59+00:00'
        timestamp=iso(raw_date)
        key=(inst,kind,source_id)
        if key in documents:
            if documents[key]['text_hash']!=digest(text):
                doc_diagnostics.append({'file':str(path),'problem':'conflicting_document_duplicate','source_id':source_id})
            continue
        documents[key]={'path':str(path),'timestamp':timestamp,'text_hash':digest(text),'kind':kind}
print('Unique PRD/meeting documents:',len(documents),'diagnostics:',len(doc_diagnostics))


Jira project files discovered: 30
Mapped Jira tickets: 9000 diagnostics: 0
Unique PRD/meeting documents: 97 diagnostics: 0


## 4. Build stable graph records
Every extracted structured item becomes a first-class node, including unresolved/partially specified items. Canonical entity references are linked **only** when an exact canonical ID is present. We do not guess endpoints from names. A stable hash of source scope + section + item index makes repeated imports idempotent. Original JSON is retained for audit.

In [5]:
nodes=[]; links=[]; issues=[]
# Repeated mentions can assert the same structural connection. Store it once.
_link_by_uid = {}
def node(uid, label, instance, **props):
    nodes.append({'uid':uid,'label':label,'instance_id':instance,**props})
def edge(uid, a, b, rel, instance, **props):
    candidate = {'uid':uid,'from_uid':a,'to_uid':b,'type':safe_type(rel),'instance_id':instance,**props}
    previous = _link_by_uid.get(uid)
    if previous is not None:
        if previous != candidate:
            raise ValueError(f'Conflicting edge ID {uid}: {previous} vs {candidate}')
        return  # exact duplicate assertion; preserve one structural edge
    _link_by_uid[uid] = candidate
    links.append(candidate)

def add_link(a,b,rel,instance,**props):
    if a==b: return
    edge('L_'+digest(a,b,rel,props.get('fact_uid'),props.get('role')),a,b,rel,instance,**props)

for e in canonical:
    uid='C_'+e['canonical_id']
    node(uid,'CanonicalEntity',e['instance_id'],canonical_id=e['canonical_id'],name=e.get('canonical_name'),entity_type=e.get('entity_type'),project_key=e.get('project_key'),payload_json=compact(e))

for ri,r in enumerate(tqdm(resolved,desc='Materializing')):
    inst=r.get('instance_id'); source=r.get('source') or {}; prov=r.get('provenance') or {}; temporal=r.get('temporal') or {}
    st=source.get('source_type'); sid=source.get('source_id'); chunk=prov.get('chunk_id')
    if not inst or not st or not sid:
        issues.append({'record':ri,'problem':'Missing source scope'}); continue
    record_uid='R_'+digest(inst,st,sid,chunk,compact(r))
    source_timestamp=first_time(source,['source_timestamp'])
    observed=first_time(source,['source_timestamp','created_at'])
    evidence_uid='E_'+digest(inst,st,sid,chunk,compact(r))
    node(record_uid,'SourceRecord',inst,source_type=st,source_id=str(sid),chunk_id=str(chunk) if chunk is not None else None,source_timestamp=source_timestamp,observed_at=observed,project_key=r.get('project_key'),payload_json=compact({'source':source,'temporal':temporal,'provenance':prov}))
    node(evidence_uid,'Evidence',inst,source_type=st,source_id=str(sid),chunk_id=str(chunk) if chunk is not None else None,source_timestamp=source_timestamp,observed_at=observed,evidence_text=prov.get('evidence_text'),payload_json=compact(prov))
    add_link(record_uid,evidence_uid,'HAS_EVIDENCE',inst)
    for entity in r.get('entities') or []:
        cid=entity.get('canonical_id')
        if cid in entity_by_id and entity_by_id[cid]['instance_id']==inst:
            add_link(record_uid,'C_'+cid,'MENTIONS',inst)
    def connect_references(obj, fact_uid):
        # Structured exact canonical-ID references only; recursive traversal preserves nested extracted fields.
        def walk(v, key=None):
            if isinstance(v,dict):
                for k,x in v.items(): walk(x,k)
            elif isinstance(v,list):
                for x in v: walk(x,key)
            elif isinstance(v,str) and v in entity_by_id and entity_by_id[v]['instance_id']==inst:
                add_link(fact_uid,'C_'+v,'REFERS_TO',inst,role=key or 'reference')
        walk(obj)
    for section,label in SECTIONS.items():
        for j,item in enumerate(r.get(section) or []):
            if not isinstance(item,dict):
                issues.append({'record':ri,'section':section,'index':j,'problem':'Non-object item'});continue
            uid='T_'+digest(inst,st,sid,chunk,compact(r),section,j)
            t=first_time(item,['event_time','timestamp','occurred_at','date','decision_date','meeting_date','blocked_at','created_at'])
            vf=first_time(item,['valid_from','effective_from','blocked_at']) or t
            vt=first_time(item,['valid_to','effective_to','unblocked_at'])
            if vf and vt and vf>vt: issues.append({'record':ri,'section':section,'index':j,'problem':'valid_from after valid_to'})
            node(uid,label,inst,source_type=st,source_id=str(sid),chunk_id=str(chunk) if chunk is not None else None,source_timestamp=source_timestamp,observed_at=observed,event_time=t,valid_from=vf,valid_to=vt,payload_json=compact(item))
            add_link(record_uid,uid,'HAS_'+label.upper(),inst)
            add_link(uid,evidence_uid,'SUPPORTED_BY',inst)
            connect_references(item,uid)
    for j,item in enumerate(r.get('relationships') or []):
        if not isinstance(item,dict):continue
        a,b=item.get('source_entity'),item.get('target_entity')
        if a not in entity_by_id or b not in entity_by_id:
            issues.append({'record':ri,'section':'relationships','index':j,'problem':'Missing canonical endpoint'});continue
        if entity_by_id[a]['instance_id']!=inst or entity_by_id[b]['instance_id']!=inst:
            issues.append({'record':ri,'section':'relationships','index':j,'problem':'Cross-instance endpoint'});continue
        uid='F_'+digest(inst,st,sid,chunk,compact(r),j)
        t=first_time(item,['timestamp','event_time','created_at'])
        vf=first_time(item,['valid_from','effective_from']) or t
        vt=first_time(item,['valid_to','effective_to'])
        if vf and vt and vf>vt: issues.append({'record':ri,'section':'relationships','index':j,'problem':'valid_from after valid_to'})
        node(uid,'Fact',inst,relation_type=safe_type(item.get('type')),source_type=st,source_id=str(sid),chunk_id=str(chunk) if chunk is not None else None,source_timestamp=source_timestamp,observed_at=observed,event_time=t,valid_from=vf,valid_to=vt,payload_json=compact(item))
        add_link(uid,'C_'+a,'FROM_ENTITY',inst)
        add_link(uid,'C_'+b,'TO_ENTITY',inst)
        add_link(uid,evidence_uid,'SUPPORTED_BY',inst)
        add_link(record_uid,uid,'ASSERTS',inst)

print('Nodes:',len(nodes),'Links:',len(links),'Review items:',len(issues))


Materializing:   0%|          | 0/5827 [00:00<?, ?it/s]

Nodes: 206634 Links: 600623 Review items: 35


## 5 — Enrich the rebuilt graph with source-backed time

This stage runs **after** deterministic graph construction but **before** export. It enriches existing canonical Issue nodes and Jira events, attaches document timestamps to their extracted SourceRecords and derived facts, and flags unknown times. It does not mutate ER inputs. Changelog event availability is derived from a matching original Jira history record, where a matching timestamp and issue key exist. Current-state Jira fields are never retroactively used as historical features.


In [6]:
from datetime import datetime, timezone
from dateutil import parser as dateparser

def norm_date(v):
    if not v:return None
    try:
        dt=dateparser.parse(str(v))
        return dt.astimezone(timezone.utc).isoformat(timespec='seconds') if dt.tzinfo else None
    except (ValueError,TypeError,OverflowError):return None

# Keep original construction IDs. Extra property fields are introduced explicitly.
source_time_by_record={}
for n in nodes:
    if n['label']!='SourceRecord':continue
    key=(n['instance_id'],str(n.get('source_type') or '').upper(),str(n.get('source_id') or ''))
    if key[1] in ('PRD','MEETING'):
        d=documents.get(key)
        if d and d['timestamp']:
            n['observed_at']=d['timestamp'];n['source_timestamp']=d['timestamp']
            n['temporal_status']='EXACT';n['temporal_basis']='original_document_header'
            source_time_by_record[key]=d['timestamp']

# Canonical Issue name is generally the Jira key; enrich only when exact key/instance matches.
issue_creation={}; issue_history_times={}
for (inst,key),ticket in jira_issues.items():
    created=norm_date(ticket.get('created_at'))
    if created: issue_creation[(inst,key)]=created
    times=set()
    for ch in ticket.get('full_changelog') or []:
        t=norm_date(ch.get('created'))
        if t:times.add(t)
    for h in ticket.get('status_history') or []:
        t=norm_date(h.get('changed_at'))
        if t:times.add(t)
    issue_history_times[(inst,key)]=times

for n in nodes:
    label=n['label'];inst=n['instance_id']
    if label=='CanonicalEntity' and n.get('entity_type')=='ISSUE':
        key=str(n.get('name') or '').upper()
        created=issue_creation.get((inst,key))
        if created:
            n['created_at']=created
            n['observed_at']=created
            n['temporal_status']='EXACT';n['temporal_basis']='jira.issue.created_at'
    elif label in ('Evidence','Event','Fact','Decision','Blocker','Dependency','ActionItem'):
        st=str(n.get('source_type') or '').upper();sid=str(n.get('source_id') or '')
        key=(inst,st,sid)
        dt=source_time_by_record.get(key)
        if dt:
            n['observed_at']=dt
            n['temporal_status']='EXACT';n['temporal_basis']='original_document_header'
        elif st=='JIRA':
            # Only Jira history events with an exact source timestamp are treated as recorded then.
            et=n.get('event_time')
            if label=='Event' and et and et in issue_history_times.get((inst,sid.upper()),set()):
                n['observed_at']=et
                n['temporal_status']='EXACT';n['temporal_basis']='jira.full_changelog.created_or_status_history.changed_at'
            elif label=='Event' and et:
                n['temporal_status']='UNKNOWN';n['temporal_basis']='event_time_without_verified_history_match'
        if not n.get('temporal_status'):
            n['temporal_status']='EXACT' if n.get('observed_at') else 'UNKNOWN'
            n['temporal_basis']='existing_source_timestamp' if n.get('observed_at') else 'unavailable'
    elif label=='SourceRecord' and not n.get('temporal_status'):
        n['temporal_status']='EXACT' if n.get('observed_at') else 'UNKNOWN'
        n['temporal_basis']='existing_source_timestamp' if n.get('observed_at') else 'unavailable'

# For document-sourced items, observation at document date does not imply that the event occurred then.
# Retain source event_time and valid_from separately. An undated Jira dependency remains UNKNOWN.
print('Timestamp enrichment complete.')
print('Canonical issues with creation dates:',sum(n['label']=='CanonicalEntity' and bool(n.get('created_at')) for n in nodes))
print('Temporal items with observed_at:',sum(n['label'] in set(SECTIONS.values())|{'Fact'} and bool(n.get('observed_at')) for n in nodes))


Timestamp enrichment complete.
Canonical issues with creation dates: 5926
Temporal items with observed_at: 87230


## 6 — Validate identity, temporal semantics and provenance

These checks are part of the notebook, not one-off Neo4j queries. Diagnostics are exported, including missing timestamps and unmatched source records. An unknown timestamp is a **reported limitation**, not automatically an integrity failure.


In [7]:
import collections
uid_lookup={n['uid']:n for n in nodes}
assert len(uid_lookup)==len(nodes),'Duplicate node UID'
assert len({e['uid'] for e in links})==len(links),'Duplicate edge UID'
assert all(e['from_uid'] in uid_lookup and e['to_uid'] in uid_lookup for e in links)
assert all(uid_lookup[e['from_uid']]['instance_id']==uid_lookup[e['to_uid']]['instance_id']==e['instance_id'] for e in links)
assert sum(n['label']=='CanonicalEntity' for n in nodes)==len(canonical)
assert all(n.get('canonical_id') in entity_by_id for n in nodes if n['label']=='CanonicalEntity')

TEMP_LABELS=set(SECTIONS.values())|{'Fact'}
coverage=[]
for label in sorted({n['label'] for n in nodes}):
    subset=[n for n in nodes if n['label']==label]
    coverage.append({'label':label,'total':len(subset),'observed_at':sum(bool(n.get('observed_at')) for n in subset),'event_time':sum(bool(n.get('event_time')) for n in subset),'temporal_unknown':sum(n.get('temporal_status')=='UNKNOWN' for n in subset)})
missing=[{'uid':n['uid'],'label':n['label'],'instance_id':n['instance_id'],'source_type':n.get('source_type'),'source_id':n.get('source_id'),'reason':n.get('temporal_basis')} for n in nodes if n['label'] in TEMP_LABELS and not n.get('observed_at')]
print('Node counts:',dict(collections.Counter(n['label'] for n in nodes)))
print('Edges:',len(links),'Unobserved temporal items:',len(missing))
for row in coverage:print(row)


Node counts: {'CanonicalEntity': 40325, 'SourceRecord': 5827, 'Evidence': 5827, 'Dependency': 7709, 'Fact': 63305, 'Event': 82577, 'Decision': 414, 'ActionItem': 207, 'Blocker': 443}
Edges: 600623 Unobserved temporal items: 67425
{'label': 'ActionItem', 'total': 207, 'observed_at': 207, 'event_time': 0, 'temporal_unknown': 0}
{'label': 'Blocker', 'total': 443, 'observed_at': 167, 'event_time': 276, 'temporal_unknown': 276}
{'label': 'CanonicalEntity', 'total': 40325, 'observed_at': 5926, 'event_time': 0, 'temporal_unknown': 0}
{'label': 'Decision', 'total': 414, 'observed_at': 414, 'event_time': 0, 'temporal_unknown': 0}
{'label': 'Dependency', 'total': 7709, 'observed_at': 237, 'event_time': 205, 'temporal_unknown': 7472}
{'label': 'Event', 'total': 82577, 'observed_at': 82396, 'event_time': 82372, 'temporal_unknown': 181}
{'label': 'Evidence', 'total': 5827, 'observed_at': 67, 'event_time': 0, 'temporal_unknown': 5760}
{'label': 'Fact', 'total': 63305, 'observed_at': 3809, 'event_tim

## 7 — Portable exports and Neo4j import scripts

All CSV fields use standard RFC-style doubled quotes and `QUOTE_ALL`. In Neo4j set `dbms.import.csv.legacy_quote_escaping=false` (restart after changing it). The notebook round-trip checks every CSV before packaging. **Import into a fresh empty database**, not on top of the previous experiment.


In [8]:
import csv,sys,collections,shutil
csv.field_size_limit(sys.maxsize)
KG_ROOT=OUTPUT_ROOT/'temporal_kg_v2_output'
for folder in ('nodes','edges','neo4j','validation','sample'):(KG_ROOT/folder).mkdir(parents=True,exist_ok=True)

def write_jsonl(path,records):
    with path.open('w',encoding='utf-8') as f:
        for record in records:f.write(compact(record)+'\n')
write_jsonl(KG_ROOT/'nodes.jsonl',nodes)
write_jsonl(KG_ROOT/'links.jsonl',links)
write_jsonl(KG_ROOT/'validation'/'review_items.jsonl',issues)
write_jsonl(KG_ROOT/'validation'/'missing_observation_times.jsonl',missing)
write_jsonl(KG_ROOT/'validation'/'jira_mapping_diagnostics.jsonl',jira_diagnostics)
write_jsonl(KG_ROOT/'validation'/'document_diagnostics.jsonl',doc_diagnostics)

NODE_COLS=['uid','instance_id','name','canonical_id','entity_type','project_key','source_type','source_id','chunk_id','source_timestamp','observed_at','event_time','valid_from','valid_to','relation_type','evidence_text','payload_json','created_at','temporal_status','temporal_basis']
EDGE_COLS=['uid','from_uid','to_uid','instance_id','role','fact_uid']

def csv_out(path,records,columns):
    with path.open('w',encoding='utf-8',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=columns,extrasaction='ignore',quoting=csv.QUOTE_ALL,doublequote=True,escapechar=None,lineterminator='\n')
        writer.writeheader()
        for row in records:writer.writerow({c:'' if row.get(c) is None else str(row[c]) for c in columns})

def csv_verify(path,expected,columns):
    with path.open('r',encoding='utf-8',newline='') as f:
        reader=csv.DictReader(f,strict=True)
        assert reader.fieldnames==columns,(path,reader.fieldnames)
        count=0
        for row in reader:
            assert None not in row and None not in row.values(),f'CSV column mismatch: {path}'
            count+=1
    assert count==expected,(path,count,expected)

node_counts=collections.Counter(n['label'] for n in nodes)
edge_counts=collections.Counter(e['type'] for e in links)
for label,count in sorted(node_counts.items()):
    p=KG_ROOT/'nodes'/f'{label}.csv'
    csv_out(p,(n for n in nodes if n['label']==label),NODE_COLS)
    csv_verify(p,count,NODE_COLS)
for rel,count in sorted(edge_counts.items()):
    p=KG_ROOT/'edges'/f'{rel}.csv'
    csv_out(p,(e for e in links if e['type']==rel),EDGE_COLS)
    csv_verify(p,count,EDGE_COLS)

import pandas as pd
pd.DataFrame(coverage).to_csv(KG_ROOT/'validation'/'temporal_coverage.csv',index=False)
report={'node_count':len(nodes),'edge_count':len(links),'node_types':dict(node_counts),'edge_types':dict(edge_counts),'jira_tickets_mapped':len(jira_issues),'original_documents_found':len(documents),'jira_mapping_issues':len(jira_diagnostics),'document_issues':len(doc_diagnostics),'unknown_temporal_observation':len(missing)}
(KG_ROOT/'validation'/'integrity_report.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print('All CSV round-trip tests passed.',len(nodes),'nodes;',len(links),'edges')


All CSV round-trip tests passed. 206634 nodes; 600623 edges


## 6. Generate local Neo4j import scripts
Copy the **contents** of `nodes/` and `edges/` to the Neo4j DBMS import directory. Run `01_constraints.cypher`, then `02_import.cypher` in Neo4j Browser (or `cypher-shell -f`). Each CSV is imported using `file:///nodes/FILENAME.csv` or `file:///edges/FILENAME.csv` if folders are copied intact. `LOAD CSV` is idempotent for nodes and edges on re-run.

In [9]:
import re
constraints=[
'CREATE CONSTRAINT kg_uid IF NOT EXISTS FOR (n:KGNode) REQUIRE n.uid IS UNIQUE;',
'CREATE INDEX kg_instance IF NOT EXISTS FOR (n:KGNode) ON (n.instance_id);',
'CREATE INDEX kg_observed IF NOT EXISTS FOR (n:KGNode) ON (n.observed_at);',
'CREATE INDEX kg_valid IF NOT EXISTS FOR (n:KGNode) ON (n.valid_from);',
'CREATE INDEX kg_canonical IF NOT EXISTS FOR (n:CanonicalEntity) ON (n.canonical_id);']
(KG_ROOT/'neo4j'/'01_constraints.cypher').write_text('\n'.join(constraints)+'\n',encoding='utf-8')
commands=[]
for label in sorted(node_counts):
    assert re.fullmatch(r'[A-Za-z][A-Za-z0-9]*',label)
    setter=', '.join(f'n.{k} = CASE WHEN row.{k} = "" THEN null ELSE row.{k} END' for k in NODE_COLS if k!='uid')
    commands.append(f"""LOAD CSV WITH HEADERS FROM 'file:///nodes/{label}.csv' AS row
CALL (row) {{
  MERGE (n:KGNode {{uid:row.uid}})
  SET n:{label}, {setter}
}} IN TRANSACTIONS OF 500 ROWS;""")
for typ in sorted(edge_counts):
    assert re.fullmatch(r'[A-Z][A-Z0-9_]*',typ)
    commands.append(f"""LOAD CSV WITH HEADERS FROM 'file:///edges/{typ}.csv' AS row
CALL (row) {{
  MATCH (a:KGNode {{uid:row.from_uid}})
  MATCH (b:KGNode {{uid:row.to_uid}})
  MERGE (a)-[r:{typ} {{uid:row.uid}}]->(b)
  SET r.instance_id = row.instance_id,
      r.role = CASE WHEN row.role = '' THEN null ELSE row.role END,
      r.fact_uid = CASE WHEN row.fact_uid = '' THEN null ELSE row.fact_uid END
}} IN TRANSACTIONS OF 500 ROWS;""")
(KG_ROOT/'neo4j'/'02_import.cypher').write_text('\n\n'.join(commands)+'\n',encoding='utf-8')
print('Import statements:',len(commands))


Import statements: 21


## 8 — Snapshot query contract and visualization examples

**Strict available-by-cutoff**: a fact must have a non-null observation timestamp on or before cutoff; its referenced entities must exist by cutoff when creation dates are known. Do not use the full graph's latest-state dependency count as a historical feature. A timestamp-free dependency is *unknown*, not automatically active. These queries are saved as `.cypher` files in the package.


In [10]:
queries={
'03_verify.cypher':"""MATCH (n:KGNode) RETURN labels(n) AS labels,count(*) AS total ORDER BY total DESC;
MATCH ()-[r]->() RETURN type(r) AS relation,count(*) AS total ORDER BY total DESC;""",
'04_visualize.cypher':"""MATCH (a:CanonicalEntity)<-[r1:FROM_ENTITY]-(f:Fact)-[r2:TO_ENTITY]->(b:CanonicalEntity)
WHERE f.instance_id = 'NEXUS-I01'
RETURN a,r1,f,r2,b LIMIT 25;

MATCH (e:Event)-[r]-(n) WHERE e.instance_id='NEXUS-I01'
RETURN e,r,n LIMIT 40;""",
'05_point_in_time.cypher':"""// Edit instance and cutoff below. Stored times are ISO-8601 UTC strings.
WITH 'NEXUS-I01' AS instance_id, '2019-03-17T00:00:00+00:00' AS cutoff
MATCH (f:Fact {instance_id:instance_id})-[:FROM_ENTITY]->(a:CanonicalEntity)
MATCH (f)-[:TO_ENTITY]->(b:CanonicalEntity)
WHERE f.observed_at IS NOT NULL AND datetime(f.observed_at) <= datetime(cutoff)
  AND (f.valid_from IS NULL OR datetime(f.valid_from) <= datetime(cutoff))
  AND (f.valid_to IS NULL OR datetime(f.valid_to) > datetime(cutoff))
  AND (a.created_at IS NULL OR datetime(a.created_at) <= datetime(cutoff))
  AND (b.created_at IS NULL OR datetime(b.created_at) <= datetime(cutoff))
RETURN f.uid AS fact_id,f.relation_type AS relation,a.canonical_id AS source,b.canonical_id AS target,
       f.observed_at AS observed_at,f.temporal_status AS temporal_status
LIMIT 1000;""",
'06_event_timeline.cypher':"""MATCH (e:Event {instance_id:'NEXUS-I01'})
WHERE e.observed_at IS NOT NULL
RETURN e.uid,e.event_time,e.observed_at,e.temporal_status,e.payload_json
ORDER BY e.observed_at LIMIT 50;"""
}
for filename,query in queries.items(): (KG_ROOT/'neo4j'/filename).write_text(query+'\n',encoding='utf-8')
# Simple project checkpoint manifest: data-driven observation windows, not copied snapshots.
from datetime import timedelta
checkpoints=[]
for inst in instance_ids:
    times=[norm_date(n.get('observed_at')) for n in nodes if n['instance_id']==inst and n.get('observed_at')]
    if not times:continue
    start=datetime.fromisoformat(min(times));end=datetime.fromisoformat(max(times));i=0
    t=start
    while t<=end:
        checkpoints.append({'instance_id':inst,'week_index':i,'cutoff':t.isoformat(timespec='seconds')})
        i+=1;t+=timedelta(days=7)
    if checkpoints[-1]['cutoff']!=end.isoformat(timespec='seconds'):
        checkpoints.append({'instance_id':inst,'week_index':i,'cutoff':end.isoformat(timespec='seconds')})
pd.DataFrame(checkpoints,columns=['instance_id','week_index','cutoff']).to_csv(KG_ROOT/'validation'/'checkpoint_manifest.csv',index=False)
print('Checkpoint rows:',len(checkpoints))


Checkpoint rows: 16701


## 9 — Manifest, reproducibility and handoff

The graph can be regenerated by running this notebook with the frozen ER inputs and original sources. The output archive includes graph JSONL, CSV, Cypher, review reports and this manifest. Local Neo4j is for validation and visualization only. **Do not claim leakage-free XGBoost readiness until the missing-availability report has been reviewed and point-in-time queries tested.**


In [11]:
import hashlib,shutil
input_hashes={}
for p in (CANONICAL_FILE,RESOLVED_FILE,MAP_FILE):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    input_hashes[p.name]=h.hexdigest()
manifest={'schema_version':'temporal-kg-v2','inputs_sha256':input_hashes,'counts':report,'timestamp_policy':'Strict known-by-cutoff; unknown availability excluded','notes':['PRD date-only interpreted conservatively at end of UTC day','Jira event observed_at derived only on matching changelog timestamp','Original source documents optional; missing inputs produce coverage diagnostics','Jira-only instance mapping requires unambiguous project key or instance folder hint']}
(KG_ROOT/'manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
(KG_ROOT/'README.md').write_text("""# Temporal KG v2 handoff

1. Check `validation/integrity_report.json`, `temporal_coverage.csv`, `missing_observation_times.jsonl`, and mapping diagnostics.
2. Start with an EMPTY local Neo4j database; set `dbms.import.csv.legacy_quote_escaping=false` and restart.
3. Copy `nodes/` and `edges/` into the Neo4j instance import folder. Run each statement in `neo4j/01_constraints.cypher`, then each in `neo4j/02_import.cypher` (nodes before edges).
4. Run `neo4j/03_verify.cypher` and visualization queries. Counts must equal the integrity report.
5. Use `neo4j/05_point_in_time.cypher` as a conservative starting contract, not a finished ML feature pipeline. Unknown-availability facts are excluded.
""",encoding='utf-8')
archive=shutil.make_archive(str(KG_ROOT),'zip',root_dir=KG_ROOT)
print('Saved:',archive)


Saved: /content/drive/MyDrive/GC1/entity_resolution_output/temporal_kg_v2_output.zip
